# FarmFederate — BLIP Image Captioning
Generate labeled agricultural text from local crop stress images using BLIP.

**Steps:**
1. Run Cell 1 (install) → **Runtime > Restart runtime**
2. Run Cell 2 (mount Drive)
3. Run Cell 3 (verify images found)
4. Run Cell 4 (generate captions — ~20-40 min on T4)

In [ ]:
# Cell 1 — Install dependencies
# After this cell finishes: Runtime > Restart runtime, then continue from Cell 2
!pip install -q transformers pillow pandas torch torchvision accelerate

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Verify image folders are found
from pathlib import Path

# ★ Change this if your Drive path is different ★
MANUAL_DATA_DIR = "/content/drive/MyDrive/FarmFederate/data"

DATA_DIR = Path(MANUAL_DATA_DIR)
STRESS_LABELS = ["water_stress", "nutrient_def", "pest_risk", "disease_risk", "heat_stress"]

print(f"DATA_DIR: {DATA_DIR}")
print(f"Exists: {DATA_DIR.exists()}\n")

total = 0
for cls in STRESS_LABELS:
    img_dir = DATA_DIR / cls / "images"
    if img_dir.exists():
        count = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
        total += count
        print(f"  {cls}: {count} images")
    else:
        print(f"  {cls}: [NOT FOUND] {img_dir}")

print(f"\nTotal images: {total}")
if total == 0:
    print("\nWARNING: No images found! Check MANUAL_DATA_DIR above.")
else:
    print("Images found — ready to generate captions.")

In [ ]:
# Cell 4 — Load BLIP model and generate captions
import random
import pandas as pd
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
MANUAL_DATA_DIR = "/content/drive/MyDrive/FarmFederate/data"  # ★ same as Cell 3 ★
MAX_PER_CLASS   = 400       # images per stress class (x2 prompts = 800 captions per class)
IMG_BATCH_SIZE  = 8         # reduce to 4 if you get CUDA out-of-memory
MODEL_ID        = "Salesforce/blip-image-captioning-large"
SEED            = 42
STRESS_LABELS   = ["water_stress", "nutrient_def", "pest_risk", "disease_risk", "heat_stress"]

# Shorter open-ended prompts so BLIP generates descriptive content rather than echoing the prefix
CLASS_PROMPTS = {
    "water_stress": [
        "a wilting crop plant with drooping leaves showing",
        "a drought stressed plant with dry curling leaves and",
    ],
    "nutrient_def": [
        "a plant leaf with yellowing chlorosis showing",
        "a pale discoloured leaf with nutrient deficiency and",
    ],
    "pest_risk": [
        "a crop leaf with insect damage and feeding holes showing",
        "a plant damaged by pests with chewed edges and",
    ],
    "disease_risk": [
        "a diseased plant leaf with fungal spots showing",
        "an infected crop leaf with blight and lesions and",
    ],
    "heat_stress": [
        "a heat stressed plant with scorched brown leaf tips showing",
        "a crop plant with heat damage wilted tips and",
    ],
}

DATA_DIR   = Path(MANUAL_DATA_DIR)
OUTPUT_CSV = DATA_DIR / "crop_stress_text_dataset.csv"
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"DATA_DIR: {DATA_DIR}")

# ── Load BLIP ─────────────────────────────────────────────────────────────────
print(f"\nLoading {MODEL_ID} ...")
processor = BlipProcessor.from_pretrained(MODEL_ID)
model = BlipForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
)
model = model.to(DEVICE).eval()
print("BLIP ready.\n")


# ── Caption function ───────────────────────────────────────────────────────────
def caption_batch(image_paths, prompt):
    images = []
    for p in image_paths:
        try:
            images.append(Image.open(p).convert("RGB"))
        except Exception:
            pass
    if not images:
        return []
    inputs = processor(
        images=images,
        text=[prompt] * len(images),
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=4,
            repetition_penalty=1.5,
            length_penalty=1.0,
        )
    return processor.batch_decode(out, skip_special_tokens=True)


# ── Generate captions ──────────────────────────────────────────────────────────
rows = []

for label_idx, stress in enumerate(STRESS_LABELS):
    img_dir = DATA_DIR / stress / "images"
    if not img_dir.exists():
        print(f"[SKIP] {img_dir} not found")
        continue

    all_paths = [str(p) for p in img_dir.glob("*.jpg")] + \
                [str(p) for p in img_dir.glob("*.png")]
    random.seed(SEED + label_idx)
    random.shuffle(all_paths)
    paths = all_paths[:MAX_PER_CLASS]

    print(f"\n{'='*60}")
    print(f"{stress}: {len(paths)} images x 2 prompts = up to {len(paths)*2} captions")
    print(f"{'='*60}")

    prompts  = CLASS_PROMPTS[stress]
    captions = []

    for pi, prompt in enumerate(prompts):
        print(f"  Prompt {pi+1}/2: \"{prompt}\"")
        for i in range(0, len(paths), IMG_BATCH_SIZE):
            batch   = paths[i : i + IMG_BATCH_SIZE]
            batch_c = caption_batch(batch, prompt)
            captions.extend(batch_c)
            done = min(i + IMG_BATCH_SIZE, len(paths))
            print(f"    {done}/{len(paths)} images done", end="\r")
        print()

    kept = 0
    for cap in captions:
        cap = cap.strip()
        if len(cap) < 30:
            continue
        rows.append({
            "text":       cap,
            "label":      label_idx,
            "label_name": stress,
            "source":     "blip_caption",
        })
        kept += 1
    print(f"  -> {kept} captions kept for {stress}")

print(f"\n{'='*60}")
print(f"Total new captions: {len(rows)}")

df_new = pd.DataFrame(rows)

# Merge: keep existing non-blip rows, replace blip rows
if OUTPUT_CSV.exists():
    df_existing = pd.read_csv(OUTPUT_CSV)
    if "source" in df_existing.columns:
        df_existing = df_existing[df_existing["source"] != "blip_caption"]
    df_out = pd.concat([df_existing, df_new], ignore_index=True)
else:
    df_out = df_new

df_out = df_out.sample(frac=1, random_state=SEED).reset_index(drop=True)
df_out.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved {len(df_out)} total rows -> {OUTPUT_CSV}")
print(df_out["label_name"].value_counts().to_string())

In [ ]:
# Cell 5 — Spot-check: print 2 sample captions per class
import pandas as pd
from pathlib import Path

MANUAL_DATA_DIR = "/content/drive/MyDrive/FarmFederate/data"
OUTPUT_CSV = Path(MANUAL_DATA_DIR) / "crop_stress_text_dataset.csv"
STRESS_LABELS = ["water_stress", "nutrient_def", "pest_risk", "disease_risk", "heat_stress"]

df = pd.read_csv(OUTPUT_CSV)
blip_only = df[df["source"] == "blip_caption"]
print(f"BLIP captions: {len(blip_only)} rows")
print(blip_only["label_name"].value_counts().to_string())

print("\n--- Sample captions per class ---")
for stress in STRESS_LABELS:
    subset = blip_only[blip_only["label_name"] == stress]["text"]
    print(f"\n[{stress}]")
    for s in subset.head(2):
        print(f"  * {s[:200]}")